# CodeQA — CodeBERT + Transformer decoder (Python)

Baseline kedua setelah Seq2seq: **encoder `microsoft/codebert-base` pretrained + decoder Transformer baru**, memakai `model.py` resmi CodeQA (commit `e0dba3ccb260447096e20ad1704013272512c779`). Decoder: 6 layer, 12 head, hidden 768, feedforward 2048, dropout 0.1, embedding target dan generator terikat dengan embedding CodeBERT. Seluruh encoder dan decoder di-fine-tune; ini bukan LoRA atau zero-shot.

**Sumber:** [CodeQA paper, Appendix B dan Table 8](https://aclanthology.org/2021.findings-emnlp.223.pdf), [source resmi](https://github.com/jadecxliu/CodeQA/tree/e0dba3ccb260447096e20ad1704013272512c779/codeBERT), [model pretrained](https://huggingface.co/microsoft/codebert-base).

| Komponen | Konfigurasi |
|---|---|
| Data | Python: 56.085 train; pilih checkpoint dari 1.000 dev tetap; test akhir 7.000 |
| Epoch / early stopping | 20 epoch, tanpa early stopping, mengikuti CodeBERT paper |
| Batch | Effective 64 (Appendix B); microbatch 4 + accumulation pada satu GPU |
| Alternatif batch | Effective 32 mengikuti `python_script.sh`; tentukan sebelum mulai run |
| Panjang / beam | Input 400, target 30 termasuk token spesial; beam 10 pada dev dan test |
| Optimizer / scheduler | AdamW LR 5e-5, eps 1e-8, weight decay 0; warmup 10% lalu linear decay |
| Adaptasi memori | FP16 + encoder gradient checkpointing; beam inference FP32 |
| Pemilihan checkpoint | Rata-rata smoothed sentence BLEU resmi; test tidak digunakan memilih model |

**Perbedaan yang perlu dilaporkan:** satu GPU + gradient accumulation menggantikan tiga GPU; precision FP16, encoder gradient checkpointing, AdamW PyTorch modern, serta koreksi batas accumulation dan pembobotan berdasarkan token. Angka identik paper tidak dijamin. Source length/target length tidak dipendekkan pada profil paper. Untuk eksperimen FP32, pilih sebelum memulai dan gunakan tag baru.

Default siap untuk percobaan pada **satu GPU 12 GB yang tersedia**. Microbatch 4 belum dibenchmark pada VPS; jika OOM, pilih 1 atau 2 sebelum training penuh dan gunakan tag baru. `smoke` masih memakai model CodeBERT asli, hanya 8 train / 4 dev / 4 test, 1 epoch dan beam 2.

## 1. Environment

Jalankan dari direktori `vpskc` (atau root branch ekspor `vpskc`). Pilih kernel Python environment VPS. Install PyTorch CUDA yang cocok terlebih dahulu; kemudian `python -m pip install -r requirements-codebert.txt`. METEOR membutuhkan Java. Download pertama encoder sekitar 500 MB dan disimpan di cache Hugging Face; checkpoint resume plus optimizer membutuhkan beberapa GB.

Tidak perlu memasang kernel bernama khusus. Metadata notebook memakai `python3`. Jika memakai GPU bersama, set `CUDA_VISIBLE_DEVICES` sebelum kernel dimulai.

In [ ]:
from pathlib import Path
import json, sys
ROOT = Path.cwd()
assert (ROOT / "codebert_training.py").is_file(), "Buka notebook dari root paket vpskc"
sys.path.insert(0, str(ROOT))
from codebert_training import Settings, run, PAPER
import torch
print("Python:", sys.executable)
print("PyTorch:", torch.__version__, "CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Free/total GPU memory:", torch.cuda.mem_get_info(0))

## 2. Konfigurasi run

Gunakan `PROFILE = "smoke"` untuk uji kecil pertama. Jika lolos, pilih `paper` dan tag yang sesuai. **Pertahankan microbatch, precision, source, dan tag saat resume.** Runner menolak konfigurasi yang berubah untuk tag yang sama. Batas waktu per sesi boleh diubah tanpa membuat identitas run baru.

Default **tidak memiliki batas dua jam**: training berjalan sampai 20 epoch selesai. Jika sengaja dihentikan, jalankan lagi konfigurasi yang sama. Satu checkpoint disimpan sekitar tiap 5 menit (pada batas effective batch) dan setelah dev; kelompok accumulation yang belum selesai tidak disimpan. Test hanya dibuka setelah training selesai.

In [ ]:
PROFILE = "paper"
RUN_TAG = "v1"
MICRO_BATCH_SIZE = 4
EFFECTIVE_BATCH_SIZE = 64  # Appendix B; 32 mengikuti skrip resmi.
EVAL_BATCH_SIZE = 1
PRECISION = "fp16"
GRADIENT_CHECKPOINTING = True
MAX_SESSION_SECONDS = 0  # 0 = sampai training selesai; boleh diubah saat resume.

settings = Settings(profile=PROFILE, run_tag=RUN_TAG, micro_batch=MICRO_BATCH_SIZE,
                    effective_batch=EFFECTIVE_BATCH_SIZE, eval_batch=EVAL_BATCH_SIZE,
                    precision=PRECISION, gradient_checkpointing=GRADIENT_CHECKPOINTING,
                    max_session_seconds=MAX_SESSION_SECONDS, resume=True)
settings

## 3. Training, pemilihan checkpoint, dan evaluasi

Cell ini melakukan seluruh alur. Log tersimpan di `codebert_paper_results/<profile>_<tag>_<hash>/training.log`, terpisah dari hasil Seq2seq. Checkpoint `last.pt` menyimpan model, AdamW, scheduler, GradScaler, cursor, dan RNG; `best.pt` menyimpan model dengan BLEU dev terbaik. Kedua file tidak masuk Git.

Saat resume, cari pesan **`RESUME confirmed`** dengan epoch dan batch yang sesuai. Setelah 10 update berhasil, `benchmark.json` memuat estimasi satu epoch **tanpa validasi/IO**. Beam 10 pada 1.000 dev per epoch juga membutuhkan waktu; waktu dev tercatat pada history. Jika OOM, traceback muncul dan checkpoint utuh terakhir dipertahankan. Jangan jalankan notebook dan runner terminal bersamaan untuk tag yang sama.

In [ ]:
RUN = run(settings)
print("Run directory:", RUN)

## 4. Membaca hasil

Skor smoke hanya memverifikasi pipeline. Bandingkan hasil paper hanya setelah 20 epoch dan evaluasi 7.000 test selesai. BLEU utama adalah rata-rata sentence BLEU resmi; corpus BLEU diberi label diagnostik. Beam upstream menggunakan FP32 agar sentinel finished-beam tidak overflow FP16. Model terbaik dimuat ulang dan hasil beam diperiksa konsistensinya. Jika Java/METEOR gagal, nilainya `null` beserta error, bukan diganti metrik lain.

In [ ]:
status = json.loads((RUN / "status.json").read_text())
print(json.dumps(status, indent=2))
if (RUN / "results.json").exists():
    results = json.loads((RUN / "results.json").read_text())
    print("Evaluation scope:", results["evaluation_scope"])
    print("metric | run | paper")
    for metric, expected in PAPER.items():
        print(metric, "|", results.get(metric), "|", expected)
else:
    print("Test belum dievaluasi. Resume dengan konfigurasi sama.")

## 5. Jalankan tanpa laptop melalui tmux

Setelah environment aktif, dari root paket:

```bash
tmux new -s codebert
python -u codebert_training.py --profile smoke --run-tag check --micro-batch 1
```

Jika smoke lolos, jalankan training penuh di sesi yang sama:

```bash
python -u codebert_training.py --profile paper --run-tag v1 --micro-batch 4
```

Tekan Ctrl+B, lepaskan, lalu D untuk detach. Laptop boleh mati selama VPS menyala. Untuk kembali: `tmux attach -t codebert`. Resume menggunakan perintah persis sama. Tidak perlu `nbconvert`, loop dua jam, atau kernel Jupyter untuk jalur terminal.

Log live dari terminal VPS lain:

```bash
tail -f codebert_paper_results/paper_v1_*/training.log
```

Pilih GPU yang tersedia sebelum mulai proses, misalnya `CUDA_VISIBLE_DEVICES=1 python -u codebert_training.py --profile paper --run-tag v1 --micro-batch 4`. Ini contoh pemilihan GPU, bukan jaminan GPU 1 kosong. Jika mengubah pilihan visible GPU, pertahankan jumlahnya satu saat resume.

Hasil baseline Seq2seq tetap menjadi pembanding terpisah. Menjalankan profil ini mereproduksi baseline CodeBERT Python, bukan semua eksperimen paper.